<a href="https://colab.research.google.com/github/D2718281828nis/LLM_agent-FireCrawl-Graph/blob/main/FireCrawl-CR-site-parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load libraries

In [22]:
# 1. Install the latest version of Firecrawl
!pip install firecrawl-py --upgrade -q

from firecrawl import Firecrawl
# Removed: from firecrawl.models import ScrapeOptions as it caused ModuleNotFoundError

Firecrawl-py version not found, possibly due to installation issue.


In [2]:
import os
from google.colab import userdata

# Retrieve the Firecrawl API key securely from Colab secrets
os.environ["FIRECRAWL_API_KEY"] = userdata.get("FIRECRAWL_API_KEY")

In [3]:
app = Firecrawl(api_key='FIRECRAWL_API_KEY')


# Check if scrap-mechanism work

In [8]:
import os
from google.colab import userdata
from firecrawl import Firecrawl

# --- Firecrawl API Key Check ---
print("Checking Firecrawl API Key...")
firecrawl_api_key = userdata.get("FIRECRAWL_API_KEY")

if not firecrawl_api_key:
    print("❌ Firecrawl API Key not found in Colab secrets. Please ensure you've added it as 'FIRECRAWL_API_KEY'.")
else:
    try:
        # Initialize Firecrawl client with the retrieved key
        app = Firecrawl(api_key=firecrawl_api_key)

        # Attempt a simple scrape to verify the key and connection
        print("Attempting a test scrape with Firecrawl...")
        # Using a reliable, publicly accessible URL for testing
        test_url = "https://www.firecrawl.com/"
        result = app.scrape(test_url)

        # The result for scrape (single URL) is typically a single Document object
        if result and hasattr(result, 'markdown') and result.markdown: # Check if it's a Document-like object with markdown content
            print("✅ Firecrawl API Key is working! Successfully scraped a test URL and retrieved content.")
            print("\n--- First 200 characters of scraped content ---")
            print(result.markdown[:200])
            print("---------------------------------------------------\n")
        elif result: # Result exists but no markdown content or not a Document-like object
            print("⚠️ Firecrawl API Key might be valid, but test scrape returned results without markdown content or an unexpected format.")
            print(f"Returned object type: {type(result)}")
        else:
            print("⚠️ Firecrawl API Key might be valid, but test scrape returned an empty or None result.")

    except Exception as e:
        print(f"❌ Firecrawl API Key check failed: {e}")
        print("Please check if your 'FIRECRAWL_API_KEY' is correct and has the necessary permissions.")


Checking Firecrawl API Key...
Attempting a test scrape with Firecrawl...
✅ Firecrawl API Key is working! Successfully scraped a test URL and retrieved content.

--- First 200 characters of scraped content ---
Introducing our most accurate /search yet. [Read the announcement →](https://www.firecrawl.dev/blog/introducing-our-most-accurate-search-yet?utm_source=firecrawl-web&utm_medium=banner&utm_campaign=int
---------------------------------------------------



# Part I

In [23]:
import pandas as pd
from firecrawl import Firecrawl
from google.colab import userdata
# Removed: from firecrawl.models import ScrapeOptions as it causes ModuleNotFoundError

# --- Firecrawl Initialization (Robust for this cell) ---
firecrawl_api_key = userdata.get("FIRECRAWL_API_KEY")
if not firecrawl_api_key:
    print("❌ Firecrawl API Key not found in Colab secrets. Please ensure you've added it as 'FIRECRAWL_API_KEY'.")
    # Exit or raise error if API key is critical for subsequent steps
    raise ValueError("Firecrawl API Key is required but not found.")

try:
    app = Firecrawl(api_key=firecrawl_api_key)
    print("✅ Firecrawl client initialized.")
except Exception as e:
    print(f"❌ Failed to initialize Firecrawl client: {e}")
    raise

# --- Scraping and Parsing Configuration ---
target_url = "https://cr.minzdrav.gov.ru/clin-rec"
output_csv_filename = "clinical_registries_minzdrav.csv"
required_headers_ru = ['ID', 'Наименование', 'Дата размещения КР', 'МКБ-10']

print(f"\nAttempting to scrape and parse data from: {target_url}")

try:
    # 1. Scrape the page using Firecrawl, ensuring JavaScript is rendered
    print("🌐 Scraping the page using Firecrawl (with page_rendered=True)...")

    # Corrected: Pass page_rendered and formats as direct keyword arguments to app.scrape
    scrape_result = app.scrape(target_url, page_rendered=True, formats=['raw_html'])

    if not scrape_result or not scrape_result.raw_html:
        print("❌ Firecrawl returned no raw HTML content.")
        print(f"Raw scrape result object: {scrape_result}")
        # If no HTML, there's nothing to parse, so exit
        raise RuntimeError("No raw HTML content received from Firecrawl.")

    raw_html = scrape_result.raw_html
    print("✅ Successfully retrieved raw HTML content from Firecrawl.")

    # 2. Parse the HTML using pandas.read_html to find tables
    print("📊 Attempting to extract tables from HTML using pandas.read_html...")
    tables = pd.read_html(raw_html)

    if not tables:
        print("❌ No HTML tables were found by pandas.read_html.")
        raise ValueError("No tables detected in the scraped HTML.")

    print(f"Found {len(tables)} potential table(s) on the page.")

    # 3. Identify the correct data table based on required headers
    data_table_df = None
    for i, table_df in enumerate(tables):
        # Clean column names (strip whitespace) for robust comparison
        # Russian column names might have leading/trailing spaces
        cleaned_columns = [col.strip() for col in table_df.columns]
        table_df.columns = cleaned_columns # Update DataFrame columns for easier access

        # Check if all required headers are present in the current table
        if all(header in cleaned_columns for header in required_headers_ru):
            data_table_df = table_df
            print(f"✅ Identified the relevant data table (Table {i+1}).")
            break

    if data_table_df is None:
        print("❌ Could not find a table containing all specified headers:")
        for header in required_headers_ru:
            print(f"   - '{header}'")
        print("\nAvailable headers from all detected tables:")
        for i, table_df in enumerate(tables):
            print(f"Table {i+1} headers: {table_df.columns.tolist()}")
        raise ValueError("Relevant data table not found.")

    # 4. Select and process the required columns
    print("✂️ Extracting specified columns and cleaning data...")
    df_output = data_table_df[required_headers_ru].copy()

    # Drop rows where essential columns like 'ID' or 'Наименование' are missing
    df_output.dropna(subset=['ID', 'Наименование'], inplace=True)

    # Convert 'ID' to string in case it's numeric and needs to be treated as such
    df_output['ID'] = df_output['ID'].astype(str)

    # 5. Save the data to a CSV file
    df_output.to_csv(output_csv_filename, index=False, encoding='utf-8')
    print(f"✅ Successfully extracted {len(df_output)} rows and saved to '{output_csv_filename}'")

    print("\n--- First 5 rows of the generated CSV content ---")
    print(df_output.head().to_markdown(index=False))
    print("-------------------------------------------------")
    print(f"CSV file saved as: {output_csv_filename}")

except Exception as e:
    print(f"An unexpected error occurred: {e}")
    import traceback
    traceback.print_exc()
    print("Consider checking the target URL manually to ensure its structure hasn't changed or if it's accessible.")

✅ Firecrawl client initialized.

Attempting to scrape and parse data from: https://cr.minzdrav.gov.ru/clin-rec
🌐 Scraping the page using Firecrawl (with page_rendered=True)...
An unexpected error occurred: FirecrawlClient.scrape() got an unexpected keyword argument 'options'
Consider checking the target URL manually to ensure its structure hasn't changed or if it's accessible.


Traceback (most recent call last):
  File "/tmp/ipykernel_22065/1446676933.py", line 33, in <cell line: 0>
    scrape_result = app.scrape(target_url, options=options_dict)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: FirecrawlClient.scrape() got an unexpected keyword argument 'options'


In [11]:
from firecrawl import Firecrawl
import re
import networkx as nx
import matplotlib.pyplot as plt


def fetch_doctor_actions_graph_firecrawl_interact(clinical_registry_id: str):
    """
    Fetches clinical registry details from minzdrav using Firecrawl's AI interaction
    and visualizes doctor actions as a graph.

    Args:
        clinical_registry_id: The ID of the clinical registry entry (e.g., '270_3')

    Returns:
        NetworkX DiGraph representing the doctor's action flow
    """
    api_key = ''  # Replace with your actual API key
    app = Firecrawl(api_key=api_key)

    base_url = f"https://cr.minzdrav.gov.ru/view-cr/{clinical_registry_id}"

    print(f"🔍 Fetching data for Clinical Registry ID: {clinical_registry_id}")
    print(f"🌐 URL: {base_url}")

    try:
        # 1. Initial scrape to start the browser session and get a scrape_id
        print("🌐 Opening the clinical registry page...")
        result = app.scrape(base_url, formats=["markdown"])

        # Extract the session ID
        if hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'):
            scrape_id = result.metadata.scrape_id
        elif isinstance(result, dict) and 'metadata' in result:
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        else:
            print("❌ Could not find scrape_id in the response.")
            return None, clinical_registry_id

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI Action: Navigate to Appendix B and extract the text
        print("📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...")
        response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача' (Doctor's Action Algorithms). "
                "Click on it to expand it if it is collapsed. "
                "Then, extract and return the full text content of this specific section. "
                "Do not summarize it, return the exact text."
            )
        )

        # The AI agent returns the extracted text in the 'output' field
        extracted_text = response.output
        print("✅ Successfully extracted the Doctor's Actions Algorithm!")

        # Display the raw content
        print("\n📄 RAW EXTRACTED CONTENT FROM FIRECRAWL INTERACT:")
        print("="*80)
        lines = extracted_text.split('\n')
        print(f"Total lines in document: {len(lines)}")
        print(f"Total characters: {len(extracted_text)}")
        print("\nFirst 50 lines of content:")
        for i, line in enumerate(lines[:50]):
            if line.strip():  # Only show non-empty lines
                print(f"{i+1:3d}: {line[:100]}{'...' if len(line) > 100 else ''}")

        if len(lines) > 50:
            print(f"\n... and {len(lines) - 50} more lines")

        print("="*80)

        # Process the text to identify steps and decisions
        steps = extract_steps_from_text(extracted_text)

        # Create graph
        G = create_action_graph(steps)

        print(f"✅ Successfully created graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

        # 3. Clean up the browser session to save credits
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return G, clinical_registry_id

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None, clinical_registry_id


def extract_steps_from_text(text_content):
    """
    Extracts steps and decisions from the clinical registry text.
    """
    # Clean up text - split into meaningful chunks
    lines = [line.strip() for line in text_content.split('\n') if line.strip()]

    steps = []
    step_number = 0

    # Pattern matching for common Russian medical step indicators
    step_patterns = [
        r'^\d+[.)]\s+',  # Numbers followed by . or )
        r'^[А-Яа-яA-Za-z]+\s*\d+[.)]\s+',  # Letter-number combinations
        r'^•\s+',  # Bullet points
        r'^-\s+',  # Dash points
        r'^[IVX]+[.)]\s+',  # Roman numerals
        r'^[А-Яа-я]{1,2}[.)]\s+',  # Single Cyrillic letters
    ]

    decision_keywords = [
        'если', 'при', 'в случае', 'когда', 'наличии', 'отсутствии',
        'if', 'when', 'in case', 'при наличии', 'при отсутствии',
        'следует', 'необходимо', 'назначить', 'провести', 'осмотреть',
        'should', 'must', 'need', 'perform', 'examine', 'prescribe'
    ]

    # More specific patterns for doctor actions
    action_keywords = [
        'следует', 'необходимо', 'назначить', 'провести', 'осмотреть', 'наблюдать', 'лечить', 'диагностировать',
        'should', 'must', 'need', 'perform', 'examine', 'prescribe', 'treat', 'diagnose'
    ]

    # Identify potential algorithm sections
    algo_start_patterns = [
        r'Приложение\s*[БB]',
        r'Алгоритмы?\s+действий?\s+врача',
        r'Действия\s+врача',
        r'Алгоритм\s+действий?',
        r'Protocol',
        r'Procedure'
    ]

    in_algorithm_section = False

    for i, line in enumerate(lines):
        # Check if we've entered an algorithm section
        line_lower = line.lower()
        if any(re.search(pattern, line, re.IGNORECASE) for pattern in algo_start_patterns):
            in_algorithm_section = True
            continue

        # If we're not in algorithm section yet, continue
        if not in_algorithm_section:
            continue

        # Skip very short lines that might be headers
        if len(line) < 15:
            continue

        # Check if this line starts a new step based on patterns
        is_new_step = any(re.match(pattern, line) for pattern in step_patterns)

        # Check if this line contains an action or decision
        has_action = any(keyword in line_lower for keyword in action_keywords)
        has_decision = any(keyword in line_lower for keyword in decision_keywords)

        # If we detect an action/decision or the line matches step pattern, process it
        if is_new_step or has_action or has_decision:
            # Clean the line from step indicators
            cleaned_line = line
            for pattern in step_patterns:
                cleaned_line = re.sub(pattern, '', cleaned_line, count=1)

            if cleaned_line.strip():  # Only add non-empty steps
                step_number += 1
                steps.append({
                    'id': step_number,
                    'text': cleaned_line.strip(),
                    'is_decision': has_decision,
                    'type': 'decision' if has_decision else 'action'
                })
        elif has_action or has_decision:
            # Even if not explicitly numbered, if it contains action keywords, add it
            step_number += 1
            steps.append({
                'id': step_number,
                'text': line.strip(),
                'is_decision': has_decision,
                'type': 'decision' if has_decision else 'action'
            })

    # If no steps were found in algorithm section, try a broader search
    if not steps:
        # Look for sentences containing action keywords in the entire text
        # Split by common sentence endings
        sentences = re.split(r'[.!?;]+', text_content)
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 25:  # Only consider substantial sentences
                has_action = any(keyword in sentence.lower() for keyword in action_keywords)
                has_decision = any(keyword in sentence.lower() for keyword in decision_keywords)

                if has_action or has_decision:
                    step_number += 1
                    steps.append({
                        'id': step_number,
                        'text': sentence,
                        'is_decision': has_decision,
                        'type': 'decision' if has_decision else 'action'
                    })

    # If still no steps, do a final broad scan for any content that looks like instructions
    if not steps:
        for line in lines:
            if len(line) > 30:  # Substantial content
                has_action = any(keyword in line.lower() for keyword in action_keywords)
                has_decision = any(keyword in line.lower() for keyword in decision_keywords)

                if has_action or has_decision or '–' in line or ':' in line:
                    step_number += 1
                    steps.append({
                        'id': step_number,
                        'text': line,
                        'is_decision': has_decision,
                        'type': 'decision' if has_decision else 'action'
                    })

    return steps


def create_action_graph(steps):
    """
    Creates a NetworkX graph from the extracted steps.
    """
    G = nx.DiGraph()

    if not steps:
        return G

    # Add nodes
    for step in steps:
        G.add_node(step['id'],
                  label=step['text'][:50] + "..." if len(step['text']) > 50 else step['text'],
                  full_text=step['text'],
                  type=step['type'])

    # Add edges - connect sequentially
    for i in range(len(steps) - 1):
        G.add_edge(steps[i]['id'], steps[i+1]['id'])

    return G


def visualize_action_graph(G, clinical_registry_id):
    """
    Visualizes the doctor's action graph.
    """
    if not G or G.number_of_nodes() == 0:
        print("No graph to visualize - no steps found in the clinical registry")
        return

    plt.figure(figsize=(16, 10))

    # Position nodes using spring layout
    pos = nx.spring_layout(G, k=3, iterations=50)

    # Separate nodes by type for different coloring
    action_nodes = [node for node, attr in G.nodes(data=True) if attr['type'] == 'action']
    decision_nodes = [node for node, attr in G.nodes(data=True) if attr['type'] == 'decision']

    # Draw nodes
    nx.draw_networkx_nodes(G, pos, nodelist=action_nodes, node_color='lightblue',
                          node_size=2500, alpha=0.8, label='Action Step')
    nx.draw_networkx_nodes(G, pos, nodelist=decision_nodes, node_color='orange',
                          node_size=2500, alpha=0.8, label='Decision Point')

    # Draw edges
    nx.draw_networkx_edges(G, pos, width=2, alpha=0.5, edge_color='gray', arrows=True, arrowsize=20)

    # Draw labels with word wrapping for better readability
    labels = {}
    for node_id in G.nodes():
        original_label = G.nodes[node_id]['label']
        # Wrap long labels
        wrapped_label = '\n'.join([original_label[i:i+20] for i in range(0, len(original_label), 20)])
        labels[node_id] = wrapped_label

    nx.draw_networkx_labels(G, pos, labels, font_size=7, font_weight='bold')

    # Add legend
    plt.legend(loc='upper right')

    plt.title(f"Doctor's Action Algorithm for Clinical Registry ID: {clinical_registry_id}", size=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Print detailed information
    print(f"\n📋 Detailed Action Steps for Clinical Registry ID: {clinical_registry_id}")
    print("=" * 80)
    for node_id in sorted(G.nodes()):
        node_data = G.nodes[node_id]
        print(f"{node_id}. [{node_data['type'].upper()}]")
        print(f"   {node_data['full_text'][:200]}{'...' if len(node_data['full_text']) > 200 else ''}")
        print("-" * 40)


def main():
    """
    Main function to run the clinical registry action analyzer using Firecrawl Interact.
    """
    print("🏥 Clinical Registry Doctor Action Analyzer (Firecrawl Interact Method)")
    print("="*80)
    print("This tool uses Firecrawl's AI interaction to access Russian Ministry of Health clinical registries")
    print("and visualizes doctor's action algorithms as graphs.")
    print("="*80)

    # Get user input for clinical registry ID
    while True:
        clinical_registry_id = input("\nEnter Clinical Registry ID (e.g., '270_3', '145'): ").strip()

        if not clinical_registry_id:
            print("⚠️  No ID provided. Please enter a valid Clinical Registry ID.")
            continue

        break

    print(f"\n🚀 Starting analysis for Clinical Registry ID: {clinical_registry_id}")

    # Fetch and process using Firecrawl Interact
    G, cid = fetch_doctor_actions_graph_firecrawl_interact(clinical_registry_id)

    if G and G.number_of_nodes() > 0:
        # Visualize the graph
        visualize_action_graph(G, cid)

        # Additional analysis
        print(f"\n📊 Graph Analysis Summary:")
        print(f"   • Total Steps: {G.number_of_nodes()}")
        print(f"   • Total Transitions: {G.number_of_edges()}")
        print(f"   • Action Steps: {len([n for n, d in G.nodes(data=True) if d['type'] == 'action'])}")
        print(f"   • Decision Points: {len([n for n, d in G.nodes(data=True) if d['type'] == 'decision'])}")

        # Check if graph is connected
        if G.number_of_nodes() > 1:
            if nx.is_weakly_connected(G):
                print("   • Flow: Sequential path exists through all steps")
            else:
                print("   • Flow: Multiple disconnected paths detected")
        else:
            print("   • Flow: Single step detected")

        return G
    else:
        print(f"❌ No actionable steps found for Clinical Registry ID: {clinical_registry_id}")
        print("💡 Possible reasons:")
        print("   • Invalid Firecrawl API key")
        print("   • Content is not publicly accessible")
        print("   • Page structure changed")
        print("   • No algorithmic steps in the registry")
        print("   • AI agent failed to locate the required section")
        return None


if __name__ == "__main__":
    main()

🏥 Clinical Registry Doctor Action Analyzer (Firecrawl Interact Method)
This tool uses Firecrawl's AI interaction to access Russian Ministry of Health clinical registries
and visualizes doctor's action algorithms as graphs.

Enter Clinical Registry ID (e.g., '270_3', '145'): 741_1

🚀 Starting analysis for Clinical Registry ID: 741_1
🔍 Fetching data for Clinical Registry ID: 741_1
🌐 URL: https://cr.minzdrav.gov.ru/view-cr/741_1
🌐 Opening the clinical registry page...
❌ An error occurred: Internal Server Error: Failed to scrape. All scraping engines failed to retrieve content from this URL. Engines tried: [index, fire-engine;chrome-cdp, fire-engine(retry);chrome-cdp]. This usually happens when: (1) The URL is invalid or the page doesn't exist (404), (2) The website is blocking automated access, (3) The website is down or unreachable, (4) The page requires authentication. Double check the URL is correct and accessible in a browser. If the issue persists, contact us at help@firecrawl.com wi

Traceback (most recent call last):
  File "/tmp/ipykernel_22065/4010650681.py", line 29, in fetch_doctor_actions_graph_firecrawl_interact
    result = app.scrape(base_url, formats=["markdown"])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/firecrawl/v2/client.py", line 234, in scrape
    return scrape_module.scrape(self.http_client, url, options)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/firecrawl/v2/methods/scrape.py", line 61, in scrape
    handle_response_error(response, "scrape")
  File "/usr/local/lib/python3.12/dist-packages/firecrawl/v2/utils/error_handler.py", line 104, in handle_response_error
    raise InternalServerError(message, response.status_code, response)
firecrawl.v2.utils.error_handler.InternalServerError: Internal Server Error: Failed to scrape. All scraping engines failed to retrieve content from this URL. Engines tried: [index, fire-engin

# Part II

In [12]:
from firecrawl import Firecrawl
import re

app = Firecrawl(api_key='FIRECRAWL_API_KEY')


def search_and_extract_with_ai(keyword: str):
    """
    AI-powered RPA workflow:
    1. Open https://cr.minzdrav.gov.ru/clin-rec
    2. Type keyword into search field under "Наименование"
    3. Click first result row → get CR ID *directly from the 'ID' column cell*
    4. Navigate to /preview-cr/{ID}#doc_b
    5. Extract full text of "Приложение Б"

    ID is extracted as raw string (preserving underscores, hyphens, prefixes).
    """
    print(f"\n🔍 Starting Firecrawl AI session for: '{keyword}'")

    try:
        # 1. Open registry
        print("🌐 Opening clinical guidelines registry...")
        result = app.scrape("https://cr.minzdrav.gov.ru/clin-rec", formats=["markdown"])

        # Extract scrape_id robustly
        scrape_id = None
        if hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'):
            scrape_id = result.metadata.scrape_id
        elif isinstance(result, dict) and 'metadata' in result:
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        if not scrape_id:
            raise RuntimeError("❌ No scrape_id in initial response")

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI: Type into search field under "Наименование"
        print(f"🔎 AI Agent typing '{keyword}' into the 'Наименование' filter...")
        app.interact(
            scrape_id,
            prompt=f"Locate the search input under the table header 'Наименование', type '{keyword}', and press Enter to filter."
        )

        # 3. AI: Click first result row → then extract ID from the *ID column cell* (leftmost)
        print("🆔 AI Agent extracting CR ID from first row's 'ID' column (raw string)...")
        id_response = app.interact(
            scrape_id,
            prompt=("""\
                The filtered table is loaded. Locate the first data row.
                Find the cell directly under the column header 'ID' (typically the leftmost column).
                Extract the *exact text content* of that cell — it may contain digits, underscores, hyphens, or letters (e.g., '230_3', '145').
                Return ONLY that raw string, no extra characters, no explanations.
                """
            )
        )
        raw_id = id_response.output.strip()

        # Sanitize: remove common noise (quotes, brackets, surrounding whitespace)
        raw_id = re.sub(r'^[\'"`\[\{]+|[\'"`\]\}]+$', '', raw_id)  # strip quotes/brackets
        cr_id = raw_id.strip()
        if not cr_id:
            raise ValueError("Empty ID extracted after cleaning")

        print(f"✅ Extracted CR ID (raw): '{cr_id}'")

        # 4. Navigate to Appendix B
        preview_url = f"https://cr.minzdrav.gov.ru/preview-cr/{cr_id}#doc_b"
        print(f"➡️ Loading: {preview_url}")
        _ = app.scrape(preview_url, formats=["markdown"])  # ensure anchor loads

        # 5. AI: Extract full text of "Приложение Б"
        print("📑 AI Agent extracting 'Приложение Б' (Doctor's Action Algorithm)...")
        algo_response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача'. "
                "If collapsed, expand it. Then select and return the FULL visible text content — verbatim, no summarization. "
                "Only return the raw text."
            )
        )
        extracted_text = algo_response.output.strip()
        if not extracted_text:
            raise RuntimeError("❌ Empty algorithm text returned")

        print("✅ Successfully extracted Doctor's Action Algorithm!")

        # 6. Cleanup
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return {
            "original_input": keyword,
            "cr_id": cr_id,
            "algorithm_text": extracted_text,
            "status": "success"
        }

    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}")
        if 'scrape_id' in locals():
            try:
                app.stop_interaction(scrape_id)
            except:
                pass
        return {
            "original_input": keyword,
            "error": str(e),
            "status": "failed"
        }


# === MAIN EXECUTION ===
if __name__ == "__main__":
    kw = input("Enter CR title (e.g., 'Прогрессирующая мышечная дистрофия'): ").strip()
    if not kw:
        print("⚠️ Empty input. Exiting.")
        exit(1)

    res = search_and_extract_with_ai(kw)

    print("\n" + "="*80)
    print("FIRECRAWL CR EXTRACTION RESULT")
    print("="*80)
    print(f"Input:          {res['original_input']}")
    if res['status'] == 'success':
        print(f"CR ID (raw):    '{res['cr_id']}'")  # preserves format: 230_3, 145-7, etc.
        print(f"Algorithm len:  {len(res['algorithm_text'])} chars")
        print("\nFirst 300 chars of Algorithm:")
        print(repr(res['algorithm_text'][:300] + ("..." if len(res['algorithm_text']) > 300 else "")))
    else:
        print(f"Error:          {res['error']}")
    print("="*80)

Enter CR title (e.g., 'Прогрессирующая мышечная дистрофия'): эпилепсия

🔍 Starting Firecrawl AI session for: 'эпилепсия'
🌐 Opening clinical guidelines registry...
❌ Error: UnauthorizedError: Unauthorized: Failed to scrape. Unauthorized: Invalid token - No additional error details provided.

FIRECRAWL CR EXTRACTION RESULT
Input:          эпилепсия
Error:          Unauthorized: Failed to scrape. Unauthorized: Invalid token - No additional error details provided.


# Part III

In [13]:
from firecrawl import Firecrawl
import re
import networkx as nx
import matplotlib.pyplot as plt
from google.colab import userdata # Import userdata to access secrets


def fetch_doctor_actions_graph_firecrawl_interact(clinical_registry_id: str):
    """
    Fetches clinical registry details from minzdrav using Firecrawl's AI interaction
    and visualizes doctor actions as a graph.

    Args:
        clinical_registry_id: The ID of the clinical registry entry (e.g., '270_3')

    Returns:
        NetworkX DiGraph representing the doctor's action flow
    """
    # Retrieve Firecrawl API key from Colab secrets
    firecrawl_api_key = userdata.get("FIRECRAWL_API_KEY")
    if not firecrawl_api_key:
        print("❌ Firecrawl API Key not found in Colab secrets. Please ensure you've added it as 'FIRECRAWL_API_KEY'.")
        return None, clinical_registry_id

    app = Firecrawl(api_key=firecrawl_api_key)

    base_url = f"https://cr.minzdrav.gov.ru/view-cr/{clinical_registry_id}"

    print(f"🔍 Fetching data for Clinical Registry ID: {clinical_registry_id}")
    print(f"🌐 URL: {base_url}")

    try:
        # 1. Initial scrape to start the browser session and get a scrape_id
        print("🌐 Opening the clinical registry page...")
        result = app.scrape(base_url, formats=["markdown"])

        print("\n--- Full raw result object from initial scrape ---")
        print(result)
        print("---------------------------------------------------\n")

        # Extract the session ID
        scrape_id = None
        if isinstance(result, list) and result:
            # If result is a list of dicts, check for 'metadata' in each item
            for item in result:
                if isinstance(item, dict) and 'metadata' in item:
                    scrape_id = item['metadata'].get('scrape_id') or item['metadata'].get('scrapeId')
                    if scrape_id: # Found it in the first item's metadata, break
                        break
        elif isinstance(result, dict) and 'metadata' in result: # For single dict result
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        elif hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'): # For object with metadata attribute
            scrape_id = result.metadata.scrape_id

        if not scrape_id:
            print("❌ Could not find scrape_id in the response.")
            # Instead of returning None and cid, try to proceed if we got content
            # and let later steps fail if scrape_id is truly needed for interact.
            # For now, if no scrape_id, we will return None, as interact needs it.
            return None, clinical_registry_id

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI Action: Navigate to Appendix B and extract the text
        print("📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...")
        response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача' (Doctor's Action Algorithms). "
                "Click on it to expand it if it is collapsed. "
                "Then, extract and return the full text content of this specific section. "
                "Do not summarize it, return the exact text."
            )
        )

        # The AI agent returns the extracted text in the 'output' field
        extracted_text = response.output if response and hasattr(response, 'output') else None

        if not extracted_text:
            print("⚠️ AI agent returned empty or invalid response")
            # Try alternative approach - extract all content from the page
            print("🔄 Attempting to extract all page content...")
            alt_response = app.interact(
                scrape_id,
                prompt="Extract all text content from the current page. Return the full text without summarization."
            )
            extracted_text = alt_response.output if alt_response and hasattr(alt_response, 'output') else None

        if not extracted_text:
            print("❌ Failed to extract any content from the page")
            app.stop_interaction(scrape_id)
            return None, clinical_registry_id

        print("✅ Successfully extracted content!")

        # Display the raw content
        print("\n📄 RAW EXTRACTED CONTENT FROM FIRECRAWL INTERACT:")
        print("="*80)
        lines = extracted_text.split('\n')
        print(f"Total lines in document: {len(lines)}")
        print(f"Total characters: {len(extracted_text)}")
        print("\nFirst 50 lines of content:")
        for i, line in enumerate(lines[:50]):
            if line.strip():  # Only show non-empty lines
                print(f"{i+1:3d}: {line[:100]}{'...' if len(line) > 100 else ''}")

        if len(lines) > 50:
            print(f"\n... and {len(lines) - 50} more lines")

        print("="*80)

        # Process the text to identify steps and decisions
        steps = extract_steps_from_markdown(extracted_text)

        # Create graph
        G = create_action_graph_improved(steps)

        print(f"✅ Successfully created graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

        # 3. Clean up the browser session to save credits
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return G, clinical_registry_id

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        import traceback
        traceback.print_exc()
        # Ensure cleanup in case of error
        try:
            if 'scrape_id' in locals() and scrape_id is not None:
                app.stop_interaction(scrape_id)
        except:
            pass
        return None, clinical_registry_id


def extract_steps_from_markdown(markdown_content):
    """
    Extracts steps and decisions from the clinical registry markdown content.
    """
    steps = []
    step_number = 0

    # Define patterns for identifying different types of content
    step_patterns = [
        r'^\d+[.)]\s+(.+)',  # Numbered lists
        r'^[А-Яа-яA-Za-z]+\s*\d+[.)]\s+(.+)',  # Letter-number combinations
        r'^•\s+(.+)',  # Bullet points
        r'^-+\s+(.+)',  # Dash points (modified to handle multiple dashes)
        r'^[IVX]+\s*[.)]\s+(.+)',  # Roman numerals
        r'^[А-Яа-я]{1,2}[.)]\s+(.+)',  # Single Cyrillic letters
    ]

    decision_keywords = [
        'если', 'при', 'в случае', 'когда', 'наличии', 'отсутствии',
        'if', 'when', 'in case', 'при наличии', 'при отсутствии',
        'следует', 'необходимо', 'назначить', 'провести', 'осмотреть'
    ]

    action_keywords = [
        'следует', 'необходимо', 'назначить', 'провести', 'осмотреть', 'наблюдать', 'лечить', 'диагностировать',
        'should', 'must', 'need', 'perform', 'examine', 'prescribe', 'treat', 'diagnose'
    ]

    # Split content into lines
    lines = markdown_content.split('\n')

    # Process markdown content to extract structured information
    current_item = ""

    for i, line in enumerate(lines):
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Check if this line starts a new step based on patterns
        is_new_step = False
        matched_text = ""

        for pattern in step_patterns:
            match = re.match(pattern, line)
            if match:
                is_new_step = True
                matched_text = match.group(1).strip()
                break

        if is_new_step:
            # Process previous item if exists
            if current_item:
                has_action = any(keyword in current_item.lower() for keyword in action_keywords)
                has_decision = any(keyword in current_item.lower() for keyword in decision_keywords)

                step_number += 1
                steps.append({
                    'id': step_number,
                    'text': current_item,
                    'is_decision': has_decision,
                    'type': 'decision' if has_decision else 'action'
                })

            # Start new item
            current_item = matched_text

        else:
            # Append to current_item if it's a continuation of the previous line (no new step indicator)
            # or if it's a standalone action/decision that wasn't caught by step_patterns
            if current_item and not any(keyword in line.lower() for keyword in action_keywords + decision_keywords):
                current_item += " " + line
            elif not current_item or (any(keyword in line.lower() for keyword in action_keywords + decision_keywords) and len(line) > 20):
                # If no current_item or if this line is an action/decision itself and substantial
                # This handles cases where steps might not be perfectly formatted with indicators but are clear actions.
                if current_item: # If there was a pending item, add it first
                    has_action = any(keyword in current_item.lower() for keyword in action_keywords)
                    has_decision = any(keyword in current_item.lower() for keyword in decision_keywords)
                    step_number += 1
                    steps.append({
                        'id': step_number,
                        'text': current_item,
                        'is_decision': has_decision,
                        'type': 'decision' if has_decision else 'action'
                    })

                current_item = line # Start a new item with this line


    # Add the last item if exists
    if current_item:
        has_action = any(keyword in current_item.lower() for keyword in action_keywords)
        has_decision = any(keyword in current_item.lower() for keyword in decision_keywords)

        step_number += 1
        steps.append({
            'id': step_number,
            'text': current_item,
            'is_decision': has_decision,
            'type': 'decision' if has_decision else 'action'
        })

    # If still no steps, try to identify by markdown headers and content structure
    if not steps:
        # Look for content after headers that might indicate algorithm sections
        header_pattern = r'^(#+)\s+(.*)$'

        in_algorithm_section = False
        for line in lines:
            # Check if this is a header
            header_match = re.match(header_pattern, line)
            if header_match:
                header_level = len(header_match.group(1))
                header_text = header_match.group(2).lower()

                # Check if this header indicates an algorithm section
                if any(keyword in header_text for keyword in ['алгоритм', 'действия врача', 'приложение б', 'protocol', 'procedure']):
                    in_algorithm_section = True
                    continue
                elif header_level <= 2:  # Major section change, likely out of algorithm
                    in_algorithm_section = False

            elif in_algorithm_section:
                # This content is within an algorithm section
                if len(line.strip()) > 20: # Ensure it's a substantial line
                    has_action = any(keyword in line.lower() for keyword in action_keywords)
                    has_decision = any(keyword in line.lower() for keyword in decision_keywords)

                    if has_action or has_decision: # Only add if it seems to be an action/decision
                        step_number += 1
                        steps.append({
                            'id': step_number,
                            'text': line.strip(),
                            'is_decision': has_decision,
                            'type': 'decision' if has_decision else 'action'
                        })

    # If still no steps, a final broad scan for any content that looks like instructions
    if not steps:
        sentences = re.split(r'[.!?;]+', markdown_content)
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 25:  # Only consider substantial sentences
                has_action = any(keyword in sentence.lower() for keyword in action_keywords)
                has_decision = any(keyword in sentence.lower() for keyword in decision_keywords)

                if has_action or has_decision:
                    step_number += 1
                    steps.append({
                        'id': step_number,
                        'text': sentence,
                        'is_decision': has_decision,
                        'type': 'decision' if has_decision else 'action'
                    })
    return steps


def create_action_graph_improved(steps):
    """
    Creates a NetworkX graph from the extracted steps with improved logic.
    """
    G = nx.DiGraph()

    if not steps:
        return G

    # Add nodes
    for step in steps:
        G.add_node(step['id'],
                  label=step['text'][:50] + "..." if len(step['text']) > 50 else step['text'],
                  full_text=step['text'],
                  type=step['type'])

    # Add edges - connect sequentially
    for i in range(len(steps) - 1):
        G.add_edge(steps[i]['id'], steps[i+1]['id'])

    # Additionally, look for conditional connections based on keywords
    # This part needs more sophisticated NLP to accurately identify branches.
    # For simplicity, we'll keep it sequential for now unless more advanced logic is requested.
    return G


def visualize_action_graph(G, clinical_registry_id):
    """
    Visualizes the doctor's action graph.
    """
    if not G or G.number_of_nodes() == 0:
        print("No graph to visualize - no steps found in the clinical registry")
        return

    plt.figure(figsize=(16, 10))

    # Position nodes using spring layout
    pos = nx.spring_layout(G, k=3, iterations=50)

    # Separate nodes by type for different coloring
    action_nodes = [node for node, attr in G.nodes(data=True) if attr['type'] == 'action']
    decision_nodes = [node for node, attr in G.nodes(data=True) if attr['type'] == 'decision']

    # Draw nodes
    nx.draw_networkx_nodes(G, pos, nodelist=action_nodes, node_color='lightblue',
                          node_size=2500, alpha=0.8, label='Action Step')
    nx.draw_networkx_nodes(G, pos, nodelist=decision_nodes, node_color='orange',
                          node_size=2500, alpha=0.8, label='Decision Point')

    # Draw edges
    nx.draw_networkx_edges(G, pos, width=2, alpha=0.5, edge_color='gray', arrows=True, arrowsize=20)

    # Draw labels with word wrapping for better readability
    labels = {}
    for node_id in G.nodes():
        original_label = G.nodes[node_id]['label']
        # Wrap long labels
        wrapped_label = '\n'.join([original_label[i:i+20] for i in range(0, len(original_label), 20)])
        labels[node_id] = wrapped_label

    nx.draw_networkx_labels(G, pos, labels, font_size=7, font_weight='bold')

    # Add legend
    plt.legend(loc='upper right')

    plt.title(f"Doctor's Action Algorithm for Clinical Registry ID: {clinical_registry_id}", size=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Print detailed information
    print(f"\n📋 Detailed Action Steps for Clinical Registry ID: {clinical_registry_id}")
    print("=" * 80)
    for node_id in sorted(G.nodes()):
        node_data = G.nodes[node_id]
        print(f"{node_id}. [{node_data['type'].upper()}]")
        print(f"   {node_data['full_text'][:200]}{'...' if len(node_data['full_text']) > 200 else ''}")
        print("-" * 40)


def main():
    """
    Main function to run the clinical registry action analyzer using Firecrawl Interact.
    """
    print("🏥 Clinical Registry Doctor Action Analyzer (Firecrawl Interact Method)")
    print("="*80)
    print("This tool uses Firecrawl's AI interaction to access Russian Ministry of Health clinical registries")
    print("and visualizes doctor's action algorithms as graphs.")
    print("="*80)

    # Get user input for clinical registry ID
    while True:
        clinical_registry_id = input("\nEnter Clinical Registry ID (e.g., '270_3', '145'): ").strip()

        if not clinical_registry_id:
            print("⚠️  No ID provided. Please enter a valid Clinical Registry ID.")
            continue

        break

    print(f"\n🚀 Starting analysis for Clinical Registry ID: {clinical_registry_id}")

    # Fetch and process using Firecrawl Interact
    G, cid = fetch_doctor_actions_graph_firecrawl_interact(clinical_registry_id)

    if G and G.number_of_nodes() > 0:
        # Visualize the graph
        visualize_action_graph(G, cid)

        # Additional analysis
        print(f"\n📊 Graph Analysis Summary:")
        print(f"   • Total Steps: {G.number_of_nodes()}")
        print(f"   • Total Transitions: {G.number_of_edges()}")
        print(f"   • Action Steps: {len([n for n, d in G.nodes(data=True) if d['type'] == 'action'])}")
        print(f"   • Decision Points: {len([n for n, d in G.nodes(data=True) if d['type'] == 'decision'])}")

        # Check if graph is connected
        if G.number_of_nodes() > 1:
            if nx.is_weakly_connected(G):
                print("   • Flow: Sequential path exists through all steps")
            else:
                print("   • Flow: Multiple disconnected paths detected")
        else:
            print("   • Flow: Single step detected")

        return G
    else:
        print(f"❌ No actionable steps found for Clinical Registry ID: {clinical_registry_id}")
        print("💡 Possible reasons:")
        print("   • Invalid Firecrawl API key")
        print("   • Content is not publicly accessible")
        print("   • Page structure changed")
        print("   • No algorithmic steps in the registry")
        print("   • AI agent failed to locate the required section")
        print("   • Extracted content didn't contain recognizable action steps")
        print("   • Content was not properly formatted as expected")
        return None


if __name__ == "__main__":
    # Comment out main() to avoid blocking input in an interactive environment
    # main()
    pass # Added pass to avoid syntax error if main() is commented out

# Call the function directly for debugging with a sample ID
print("\n--- Debugging Firecrawl response with ID '145' ---")
G_debug, cid_debug = fetch_doctor_actions_graph_firecrawl_interact('145')

if G_debug and G_debug.number_of_nodes() > 0:
    print(f"✅ Successfully fetched graph for ID {cid_debug} during debug.")
else:
    print(f"❌ Failed to fetch graph for ID {cid_debug} during debug. Check previous output for errors.")



--- Debugging Firecrawl response with ID '145' ---
🔍 Fetching data for Clinical Registry ID: 145
🌐 URL: https://cr.minzdrav.gov.ru/view-cr/145
🌐 Opening the clinical registry page...

--- Full raw result object from initial scrape ---
markdown='- Главная\n- /\n- Загрузка документа...\n\n\n© 2026 Министерство здравоохранения Российской Федерации\n\nВсе материалы, находящиеся на сайте, охраняются в соответствии с законодательством\n\nРоссийской Федерации, в том числе об авторском праве и смежных правах\n\n[Политика конфиденциальности](https://cr.minzdrav.gov.ru/view-cr/145)\n\nСкачайте приложение\n\n![](https://cr.minzdrav.gov.ru/assets/googleplay-B8tW3BE-.svg)\n\n![](https://cr.minzdrav.gov.ru/assets/appstore-CmLGhvDG.svg)\n\n![](https://cr.minzdrav.gov.ru/assets/nashstore-Ci7fMNQQ.svg)' html=None raw_html=None json=None summary=None metadata=DocumentMetadata(title='Просмотр КР', description=None, url='https://cr.minzdrav.gov.ru/view-cr/145', language='ru', keywords=None, robots=None, 

KeyboardInterrupt: 